# Chapter 33: M-current PING and PINB

Two representative 40 E/10 I networks use the source-small population size. First, M-current PING with $g_M=0.5$; second, a PINB-style comparison with no adaptation and distinct fast I→I (9 ms) and slow I→E (20 ms) synaptic pools. The q and s ODEs are separately integrated for both inhibitory outputs. Near-rest initialization differs from the source phase-splay examples; Gaussian drive heterogeneity uses the source 5% scale. Gap junctions, assembly variants and parameter sweeps are omitted.

Reference: [Python chapter](../python/chapter33.ipynb) and [chapter guide](../python/chapter33.md). The Brian2 implementation uses explicit clocks and continuous ODEs; it does not execute the reference notebook. Numerical voltage/current values use the source chapter conventions (mV/uA), and time is in ms.

Run from a local source checkout with dependencies installed (`uv sync`). These examples do not install an unreleased package from PyPI.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "mnd").is_dir():
    root = root.parent
if not (root / "mnd").is_dir():
    raise RuntimeError("Run from this repository or its brian directory; install the checkout with uv sync first.")
sys.path.insert(0, str(root))
import brian2 as b2
import numpy as np
import matplotlib.pyplot as plt
output = root / "results" / "brian-expansion"
output.mkdir(parents=True, exist_ok=True)
from mnd.brian.networks import simulate_ping


In [ ]:
rng = np.random.default_rng(63806)
current_e = 3.*(1+.05*rng.standard_normal(40))
current_i = .7*(1+.05*rng.standard_normal(10))
m = simulate_ping(current_e=current_e, current_i=current_i, g_m=.5, duration=200.)
r = simulate_ping(current_e=current_e, current_i=current_i, g_m=0., tau_ie=20., tau_ii=9., duration=200.)
assert not np.allclose(r.i_state.s_ie, r.i_state.s_ii)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(r.e_spikes.t/b2.ms, r.e_spikes.i, s=5, label='E')
ax.scatter(r.i_spikes.t/b2.ms, r.i_spikes.i+len(r.e), s=5, label='I')
ax.set(xlabel='Time (ms)', ylabel='Cell index')
ax.legend()
fig.tight_layout()
fig.savefig(output/'chapter33-pinb.png')
fig, ax = plt.subplots()
ax.plot(r.i_state.t/b2.ms, r.i_state.s_ie[0], label='I to E')
ax.plot(r.i_state.t/b2.ms, r.i_state.s_ii[0], label='I to I')
ax.set(xlabel='Time (ms)',ylabel='Synaptic gate'); ax.legend()
fig.savefig(output/'chapter33-gate-pools.png')
